##Persiapan Praktikum


In [1]:
!pip install faker -q

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA = '/content/data'
DIR_SIMPAN = '/content/drive/MyDrive/BigData/Praktikum2'
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['transaksi_mentah.csv', 'transaksi_bersih.csv']


##K-1. Import Library dan Inisialisasi
Dilakukan import terhadap library yang akan digunakan dalam praktikum, yaitu numpy, pandas, faker, dan random.

In [3]:
import numpy as np
import pandas as pd
from faker import Faker
import random


##Latihan 1 Ubah SEED menjadi 7 dan jalankan ulang seluruh pipeline
Diubah nilai konstanta SEED dari 42 menjadi 7 pada sel pembangkitan data, sehingga nilai tersebut ikut diteruskan ke np.random.seed(), random.seed(), dan Faker.seed(), serta ke argumen random_state pada seluruh pemanggilan df.sample().

Dijalankan ulang seluruh sel dari atas ke bawah, lalu dicatat jumlah baris transaksi_mentah.csv melalui len(df), jumlah nilai kosong setiap kolom melalui df.isnull().sum(), jumlah baris duplikat melalui df.duplicated().sum(), dan jumlah baris transaksi_bersih.csv setelah pembersihan, untuk dibandingkan dengan keluaran notebook yang menggunakan nilai 42 sebagai random seed.

In [4]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


##K-3. Deteksi dan Penanganan Missing Value
Dilakukan pemeriksaan keterisian data melalui df.isnull() yang mengubah setiap sel menjadi nilai boolean bernilai True untuk sel kosong, lalu hasilnya dijumlahkan per kolom melalui sum() sehingga dicetak Series berisi nama setiap kolom beserta banyaknya nilai kosong pada kolom tersebut.

In [5]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


##K-3. Deteksi dan Penanganan Missing Value
Proses pembersihan missing values dilakukan dengan menghapus baris yang kosong secara spesifik pada kolom customer_name dan payment_method menggunakan dropna(), serta mengisi nilai null pada shipping_city dengan label "Tidak Diketahui" melalui fillna(). Setelah penanganan tersebut, dataset akhirnya menyisakan 490 baris data siap pakai, sementara kolom rating yang memiliki jumlah nilai kosong terbanyak tetap dipertahankan

In [6]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah drop_dropna(): ", len(df))

Jumlah baris setelah drop_dropna():  495


##K-4. Deteksi dan Penanganan Duplicate
Dicetak jumlah baris duplikat sempurna melalui df.duplicated().sum() yang menandai baris kedua dan seterusnya apabila seluruh nilai kolomnya sama persis, lalu dicetak pula jumlah identitas transaksi yang berulang melalui df['transaction_id'].duplicated().sum() sebagai pemeriksaan tambahan pada kolom yang seharusnya unik.
Baris duplikat dibuang melalui df.drop_duplicates() yang mempertahankan kemunculan pertama setiap baris, hasilnya ditugaskan kembali ke df, lalu jumlah baris yang tersisa dicetak melalui len(df) sebagai pembanding terhadap jumlah baris sebelum penghapusan. Setelah program dijalankan, maka dapat diketahui bahwa seluruh terdapat 15 data duplikat pada seluruh dataset. Setelah data duplikat dihapus, tersisa 500 baris pada dataset


In [7]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate: ", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates(): ", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate:  5
Jumlah baris setelah drop_duplicates():  490


##K-5. Koreksi Tipe Data dan Standardisasi Format
a. Standardisasi teks kategorikal (category, payment_method, shipping_city):
Kolom teks category, payment_method, dan shipping_city distandardisasi dengan menghapus spasi berlebih dan menyeragamkan huruf awal menjadi kapital menggunakan fungsi strip() dan title(). Khusus pada kolom payment_method, nilai "Cod" dikoreksi kembali menjadi "COD" melalui fungsi replace() agar penulisan singkatan tersebut tetap sesuai dengan aslinya

In [8]:
for col in ["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

##K-5. Koreksi Tipe Data dan Standardisasi Format
b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik)

Proses penyeragaman format pada kolom price dilakukan dengan mendefinisikan sebuah fungsi khusus bernama bersihkan_harga(x). Fungsi ini bekerja dengan terlebih dahulu mengecek nilai kosong menggunakan pd.isna() agar sel yang null dapat dilewati dan langsung dikembalikan sebagai np.nan. Jika berisi data, nilai tersebut diubah menjadi string untuk dibersihkan menggunakan fungsi strip() guna membuang spasi di kedua tepinya, dilanjutkan dengan tiga tahap replace() berurutan untuk menghapus awalan "Rp", menghilangkan titik pemisah ribuan, dan mengubah koma menjadi titik agar sesuai dengan standar format desimal pada Python.

Setelah teks berhasil dibersihkan, nilai tersebut dikonversi menjadi bilangan pecahan (float) di dalam blok try-except. Penggunaan metode penanganan galat ini memastikan bahwa jika ada nilai yang tidak valid dan memicu ValueError, sistem akan dengan aman mengubahnya menjadi np.nan tanpa menghentikan eksekusi keseluruhan program. Terakhir, fungsi ini diterapkan ke seluruh baris pada kolom price menggunakan metode apply(), sehingga keempat variasi penulisan harga yang berantakan kini menyatu menjadi satu kolom bertipe numerik yang seragam.

In [9]:
def bersihkan_harga(x):
  if pd.isna(x):
    return np.nan
  x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
  try:
    return float(x)
  except ValueError:
    return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

##K-5. Koreksi Tipe Data dan Standardisasi Format
c. Standardisasi format tanggal ke YYYY-MM-DD:

Penyeragaman format pada kolom transaction_date dilakukan dengan membuat fungsi parse_tanggal(x) yang mengiterasi tiga pola penulisan tanggal, yaitu %Y-%m-%d, %d/%m/%Y, dan %d-%m-%Y. Melalui blok try-except, setiap pola diuji coba untuk mengonversi nilai menjadi objek tanggal menggunakan pd.to_datetime() dengan parameter format secara eksplisit. Penggunaan parameter ini sangat penting guna mencegah sistem keliru menukar posisi angka hari dan bulan. Jika semua pola gagal dicocokkan, fungsi secara otomatis akan mengembalikan pd.NaT (Not a Time) sebagai penanda data yang tidak sah tanpa menghentikan eksekusi program.

Setelah fungsi tersebut diterapkan pada kolom transaction_date menggunakan apply(), seluruh objek tanggal yang berhasil diekstrak kemudian diseragamkan kembali ke dalam satu format baku (YYYY-MM-DD) melalui metode .dt.strftime("%Y-%m-%d"). Hasil akhir ini ditugaskan kembali ke kolom yang sama, sehingga variasi penulisan tanggal yang sebelumnya berantakan kini menjadi seragam, konsisten, dan siap untuk dianalisis.

In [10]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

##K-5. Koreksi Tipe Data dan Standardisasi Format
d. Finalisasi tipe data:

Dikoreksi tipe data kedua kolom numerik, yaitu kolom quantity dikonversi ke bilangan bulat melalui astype(int) dan kolom price dikonversi ke bilangan pecahan melalui astype(float), lalu masing-masing hasilnya di-assign kembali ke kolom yang sama.

In [11]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

##K-6. Ekspor Dataset Bersih
DataFrame di-export menjadi file transaksi_bersih.csv melalui to_csv() berargumen index=False, sehingga indeks tidak ikut tertulis sebagai kolom. Jumlah barisnya kemudian dicetak melalui len(df).

In [12]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan: ", len(df), "baris")

Dataset bersih tersimpan:  490 baris


In [13]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating
0,TRX00490,"Baktiadi Napitupulu, S.H.",Aspernatur Basic,Rumah Tangga,120000.0,2,Transfer Bank,2026-09-11,Bau-Bau,1.0
1,TRX00430,Faizah Kusumo,Voluptates Max,Olahraga,250000.0,3,Transfer Bank,2026-07-14,Cilegon,4.0
2,TRX00083,"Drs. Sari Aryani, M.TI.",Deserunt Basic,Olahraga,75000.0,5,Kartu Kredit,2026-09-06,Tangerang Selatan,1.0
3,TRX00121,"Ilsa Mahendra, S.T.",Quia Pro,Elektronik,500000.0,1,Transfer Bank,2026-08-13,Bengkulu,4.0
5,TRX00148,Cahyanto Agustina,Officia Basic,Buku,15000.0,5,Transfer Bank,2026-08-11,Tidore Kepulauan,4.0


##Latihan 2: Tambahkan kolom is_valid_price bernilai True jika price > 0. Gunakan untuk memeriksa apakah ada harga tidak valid
Dibentuk kolom baru is_valid_price melalui np.where() atas kondisi df["price"] > 0, yang bernilai True untuk harga positif dan False untuk selainnya, lalu lima baris pertama ditampilkan dengan df.head() untuk memastikan kolom tersebut terbentuk.

Dicetak sebaran nilai kolom tersebut melalui df["is_valid_price"].value_counts(), dan seluruh 490 baris tercatat bernilai True sehingga tidak ditemukan harga tidak sah pada dataset bersih.


In [14]:
df["is_valid_price"] = np.where(df["price"] > 0, True, False)
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating,is_valid_price
0,TRX00490,"Baktiadi Napitupulu, S.H.",Aspernatur Basic,Rumah Tangga,120000.0,2,Transfer Bank,2026-09-11,Bau-Bau,1.0,True
1,TRX00430,Faizah Kusumo,Voluptates Max,Olahraga,250000.0,3,Transfer Bank,2026-07-14,Cilegon,4.0,True
2,TRX00083,"Drs. Sari Aryani, M.TI.",Deserunt Basic,Olahraga,75000.0,5,Kartu Kredit,2026-09-06,Tangerang Selatan,1.0,True
3,TRX00121,"Ilsa Mahendra, S.T.",Quia Pro,Elektronik,500000.0,1,Transfer Bank,2026-08-13,Bengkulu,4.0,True
5,TRX00148,Cahyanto Agustina,Officia Basic,Buku,15000.0,5,Transfer Bank,2026-08-11,Tidore Kepulauan,4.0,True


In [16]:
print(df["is_valid_price"].value_counts())

is_valid_price
True    490
Name: count, dtype: int64


##Latihan 3: Hitung jumlah transaksi per category menggunakan value_counts() pada dataset yang sudah bersih

Dibaca kembali berkas transaksi_bersih.csv ke DataFrame df_bersih melalui pd.read_csv(), lalu dicetak jumlah kategori unik melalui nunique() dan jumlah transaksi setiap kategori melalui value_counts().

In [15]:
df_bersih = pd.read_csv("transaksi_bersih.csv")
print("Jumlah kategori pada dataset: ", df_bersih["category"].nunique())
print(df_bersih["category"].value_counts())

Jumlah kategori pada dataset:  6
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: int64
